In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-2-4-human-benchmark-MLP

Run human age-prediction MLP benchmarks for selected gene sets.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


Adapted from the mouse MLP benchmark: predict human age using selected genes and `StandardScaler + MLPRegressor(tanh)`.

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import sys
import re
import random
import hashlib
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================

# ============================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 1))
GENE_SELECTION_THRESHOLDS = [5, 10, 15, 20, 25]
NUM_RANDOM_SAMPLES = 100
TARGET_RUN_IDX = 5
SAGE_TARGET_FOLD = "fold_0"

BENCHMARK_GENE_DIR = Path(input_path("1-human-benchmark-model-output/0-gene-database"))
HUMAN_H5_BASE_DIR = Path(input_path("1-human-benchmark-model-output/3-human-tissue-filter-remove-AL-all-tissue"))
HUMAN_HEADER_FILE = Path(input_path("2-8.3-shanda/1-data/GSE201333_RAW/gene_names_no_clones.txt"))
OUTPUT_DIR = Path(input_path("1-human-benchmark-model-output/summary_age_prediction/mlp_tanh"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HUMAN_LABEL_TO_YEARS = {
    0: 22,
    1: 33,
    2: 37,
    3: 38,
    4: 40,
    5: 42,
    6: 46,
    7: 56,
    8: 57,
    9: 59,
    10: 61,
    11: 67,
    12: 69,
    13: 74,
}

MLP_TANH_PARAMS = {
    "hidden_layer_sizes": (64, 32),
    "activation": "tanh",
    "alpha": 0.01,
    "max_iter": 500,
    "early_stopping": True,
}

MODEL_CONFIGS = {
    "scimmuaging": {
        "base_output_dir": Path(input_path("1-human-benchmark-model-output/4-scimmuaging/output_5x")),
        "file_pattern": "{label}_Fold{run_idx}_rankproduct_gene.txt",
        "dir_pattern": "Fold_{run_idx}",
    },
    "buckley": {
        "base_output_dir": Path(input_path("1-human-benchmark-model-output/5-buckley/1-output")),
        "file_pattern": "{label}_aging_related_genes_fold{run_idx}.csv",
        "dir_pattern": "",
    },
    "scale": {
        "base_output_dir": Path(input_path("1-human-benchmark-model-output/3-scale/kfold_results-split_by_tissue")),
        "file_pattern": "initial_genes_fold{run_idx}_mapped.csv",
        "dir_pattern": "Fold_{run_idx}",
    },
    "sage": {
        "base_output_dir": HUMAN_H5_BASE_DIR,
        "file_pattern": "feature.txt",
        "dir_pattern": f"cv_folds/{SAGE_TARGET_FOLD}/run",
        "header_path": HUMAN_HEADER_FILE,
    },
    "XGboost": {
        "base_output_dir": Path(input_path("1-human-benchmark-model-output/1-XGBoost_Paper/1-output")),
        "file_pattern": "xist_genes.csv",
        "dir_pattern": "Fold_{run_idx}",
    },
    "iage": {
        "base_output_dir": Path(input_path("1-human-benchmark-model-output/2-iAge/2-map-output")),
        "file_pattern": "{label}_iAge_CV_Genes_Mapped.csv",
        "dir_pattern": "",
    },
}

# ============================================================

# ============================================================
def normalize_gene(gene):
    if pd.isna(gene):
        return ""
    return str(gene).strip().upper()


def tissue_label_from_dir(tissue_dir_name):
    name = re.sub(r"^type_\d+_", "", str(tissue_dir_name))
    return name.replace("_", " ").title().replace(" ", "_")


def labels_to_years(label_array):
    label_array = np.asarray(label_array)
    raw = label_array[:, 0].astype(int) if label_array.ndim > 1 else label_array.astype(int)
    mapped = np.array([HUMAN_LABEL_TO_YEARS.get(int(x), np.nan) for x in raw], dtype=float)
    if np.isnan(mapped).any():
        bad = sorted(set(raw[np.isnan(mapped)]))
        raise ValueError(f"Unmapped human age labels found: {bad}")
    return mapped


def load_header_genes():
    if not HUMAN_HEADER_FILE.exists():
        raise FileNotFoundError(f"Human header not found: {HUMAN_HEADER_FILE}")
    genes = [normalize_gene(x) for x in HUMAN_HEADER_FILE.read_text().splitlines() if x.strip()]
    return genes, {g: i for i, g in enumerate(genes)}


def stable_seed(*parts):
    seed_str = "_".join(map(str, parts))
    return int(hashlib.md5(seed_str.encode("utf-8")).hexdigest(), 16) % (2**32)


def build_age_predictor(random_state):
    return MLPRegressor(**MLP_TANH_PARAMS, random_state=random_state)


def evaluate_age_prediction(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    if len(y_true) > 1 and np.std(y_true) > 1e-8 and np.std(y_pred) > 1e-8:
        r, _ = pearsonr(y_true, y_pred)
    else:
        r = np.nan
    return r, mae, rmse


def calculate_precision(predicted_genes_set, reference_set):
    if len(predicted_genes_set) == 0:
        return 0.0, 0
    matched = predicted_genes_set.intersection(reference_set)
    return len(matched) / len(predicted_genes_set), len(matched)

# ============================================================
# 3. benchmark gene sets
# ============================================================
def load_benchmark_genes(filepath):
    ext = filepath.suffix.lower()
    try:
        if ext == ".xlsx":
            df = pd.read_excel(filepath)
        elif ext == ".csv":
            df = pd.read_csv(filepath)
        else:
            return None

        candidates = [
            "Symbol", "symbol", "Gene", "gene", "Gene symbol", "Gene Symbol",
            "Aging_map", "names", "Name"
        ]
        gene_col = next((c for c in candidates if c in df.columns), None)
        if gene_col is None:
            gene_col = df.columns[0]
        return set(df[gene_col].dropna().map(normalize_gene).tolist())
    except Exception as e:
        print(f"跳过 benchmark 文件 {filepath}: {e}")
        return None


def load_all_benchmarks():
    if not BENCHMARK_GENE_DIR.exists():
        raise FileNotFoundError(f"Benchmark gene dir not found: {BENCHMARK_GENE_DIR}")
    out = {}
    for path in sorted(BENCHMARK_GENE_DIR.iterdir()):
        if path.is_file() and path.suffix.lower() in [".csv", ".xlsx"]:
            genes = load_benchmark_genes(path)
            if genes:
                out[path.name] = genes
    if not out:
        raise RuntimeError("未加载到任何人类 benchmark gene set。")
    return out

# ============================================================
# 4. model gene extraction
# ============================================================
def read_csv_gene_list(path, model_name):
    df = pd.read_csv(path)
    if df.empty:
        return []

    if model_name == "scale":
        df.columns = df.columns.str.lower().str.replace(".", "_", regex=False)
        genes = []
        for col in ["up", "down"]:
            if col in df.columns:
                genes.extend(df[col].dropna().map(normalize_gene).tolist())
        if genes:
            return genes

    gene_col = None
    for col in df.columns:
        col_lower = str(col).lower()
        if col_lower in ["gene", "symbol", "gene_symbol", "gene name", "gene_symbol_mapped", "features", "feature"]:
            gene_col = col
            break
    if gene_col is None:
        for col in df.columns:
            col_lower = str(col).lower()
            if "gene" in col_lower or "symbol" in col_lower:
                gene_col = col
                break
    if gene_col is None:
        gene_col = df.columns[0]

    score_col = None
    for col in df.columns:
        col_lower = str(col).lower()
        if any(k in col_lower for k in ["importance", "score", "weight", "coefficient"]):
            score_col = col
            break
    if score_col is not None:
        df = df.drop_duplicates(subset=[gene_col]).sort_values(score_col, ascending=False)

    return df[gene_col].dropna().map(normalize_gene).tolist()


def get_sage_gene_pool(tissue_dir, threshold, header_genes):
    run_dir = HUMAN_H5_BASE_DIR / tissue_dir / MODEL_CONFIGS["sage"]["dir_pattern"]
    if not run_dir.exists():
        return [], 2

    feature_dirs = []
    for d in run_dir.iterdir():
        if d.is_dir():
            m = re.match(r"feature(\d+)", d.name)
            if m:
                feature_dirs.append((int(m.group(1)), d))
    feature_dirs.sort(key=lambda x: x[0])

    selected_dir = None
    for n_genes, d in feature_dirs:
        if n_genes >= threshold:
            selected_dir = d
            break
    if selected_dir is None:
        return [], 1

    feature_file = selected_dir / "feature.txt"
    if not feature_file.exists():
        return [], 2
    try:
        mask = [float(x.strip()) for x in feature_file.read_text().splitlines() if x.strip()]
        if len(mask) != len(header_genes):
            return [], 2
        genes = [header_genes[i] for i, val in enumerate(mask) if val == 1.0]
        return list(dict.fromkeys(genes)), 0
    except Exception:
        return [], 2


def get_model_gene_pool(model_name, config, tissue_label, tissue_dir, threshold, header_genes):
    if model_name == "sage":
        return get_sage_gene_pool(tissue_dir, threshold, header_genes)

    if model_name == "scimmuaging":
        file_path = config["base_output_dir"] / tissue_label / config["dir_pattern"].format(run_idx=TARGET_RUN_IDX) / config["file_pattern"].format(label=tissue_label, run_idx=TARGET_RUN_IDX)
    elif model_name == "buckley":
        file_path = config["base_output_dir"] / tissue_label / config["file_pattern"].format(label=tissue_label, run_idx=TARGET_RUN_IDX)
    elif model_name in ["scale", "XGboost"]:
        file_path = config["base_output_dir"] / tissue_label / config["dir_pattern"].format(run_idx=TARGET_RUN_IDX) / config["file_pattern"].format(run_idx=TARGET_RUN_IDX)
    elif model_name == "iage":
        file_path = config["base_output_dir"] / config["file_pattern"].format(label=tissue_label)
    else:
        return [], 2

    if not file_path.exists():
        return [], 2
    try:
        if file_path.suffix.lower() == ".txt":
            genes = [normalize_gene(x) for x in file_path.read_text().splitlines() if x.strip()]
        elif file_path.suffix.lower() == ".csv":
            genes = read_csv_gene_list(file_path, model_name)
        else:
            return [], 2
        genes = list(dict.fromkeys([g for g in genes if g]))
        if len(genes) < threshold:
            return genes, 1
        return genes, 0
    except Exception as e:
        print(f"读取失败 {model_name} {tissue_label}: {e}")
        return [], 2

# ============================================================
# 5. age prediction by tissue, parallelized
# ============================================================
def process_one_tissue_age_prediction(tissue_dir, tissue_label, prediction_pools_cache, insufficient_set, gene_to_idx):
    keys = [k for k in prediction_pools_cache if k[1] == tissue_label]
    if not keys:
        return {}

    train_h5 = HUMAN_H5_BASE_DIR / tissue_dir / "initial_split" / "train.h5"
    test_h5 = HUMAN_H5_BASE_DIR / tissue_dir / "initial_split" / "test.h5"
    if not train_h5.exists() or not test_h5.exists():
        return {}

    try:
        with h5py.File(train_h5, "r") as f:
            x_train_all = f["data"][:].astype(np.float32)
            y_train = labels_to_years(f["label"][:])
        with h5py.File(test_h5, "r") as f:
            x_test_all = f["data"][:].astype(np.float32)
            y_test = labels_to_years(f["label"][:])

        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train_all)
        x_test_scaled = scaler.transform(x_test_all)
    except Exception as e:
        print(f"跳过 {tissue_label}: H5 读取或标准化失败: {e}")
        return {}

    out = {}
    for key in keys:
        model_name, _, threshold = key
        if (tissue_label, threshold) in insufficient_set:
            continue
        genes_pool = prediction_pools_cache.get(key, [])
        valid_indices = [gene_to_idx[g] for g in genes_pool if g in gene_to_idx]

        pr_avg, mae_avg, rmse_avg = np.nan, np.nan, np.nan
        if valid_indices:
            if len(valid_indices) > threshold:
                seed = stable_seed("human_mlp_regression", model_name, tissue_label, threshold)
                rng = random.Random(seed)
                pr_scores, mae_scores, rmse_scores = [], [], []
                sampled_index_sets = [rng.sample(valid_indices, threshold) for _ in range(NUM_RANDOM_SAMPLES)]
                for sampled_idx in sampled_index_sets:
                    model = build_age_predictor(seed)
                    model.fit(x_train_scaled[:, sampled_idx], y_train)
                    pred = model.predict(x_test_scaled[:, sampled_idx])
                    pr, mae, rmse = evaluate_age_prediction(y_test, pred)
                    if not np.isnan(pr):
                        pr_scores.append(pr)
                    mae_scores.append(mae)
                    rmse_scores.append(rmse)
                pr_avg = float(np.mean(pr_scores)) if pr_scores else np.nan
                mae_avg = float(np.mean(mae_scores)) if mae_scores else np.nan
                rmse_avg = float(np.mean(rmse_scores)) if rmse_scores else np.nan
            else:
                seed = stable_seed("human_mlp_fixed", model_name, tissue_label, threshold)
                model = build_age_predictor(seed)
                model.fit(x_train_scaled[:, valid_indices], y_train)
                pred = model.predict(x_test_scaled[:, valid_indices])
                pr_avg, mae_avg, rmse_avg = evaluate_age_prediction(y_test, pred)

        out[key] = {"Pearson_R": pr_avg, "MAE": mae_avg, "RMSE": rmse_avg}

    print(f"完成组织: {tissue_label}, 组合数: {len(out)}")
    return out

# ============================================================
# 6. main benchmark
# ============================================================
def main():
    header_genes, gene_to_idx = load_header_genes()
    benchmarks = load_all_benchmarks()

    tissue_dir_to_label = {
        p.name: tissue_label_from_dir(p.name)
        for p in HUMAN_H5_BASE_DIR.iterdir()
        if p.is_dir() and (p / "initial_split" / "train.h5").exists() and (p / "initial_split" / "test.h5").exists()
    }
    label_to_dir = {v: k for k, v in tissue_dir_to_label.items()}
    tissue_labels = sorted(label_to_dir)

    all_raw_results = []
    insufficient_set = set()
    successfully_processed = set()
    prediction_pools_cache = {}

    print(f"Loaded benchmark gene sets: {list(benchmarks)}")
    print(f"Human tissues with H5 data: {len(tissue_labels)}")
    print("\n--- Stage 1: 解析模型基因并计算 Precision ---")

    for model_name, config in MODEL_CONFIGS.items():
        print(f"Model: {model_name}")
        for tissue_label in tissue_labels:
            tissue_dir = label_to_dir[tissue_label]
            for threshold in GENE_SELECTION_THRESHOLDS:
                pool_cached = False
                for benchmark_name, benchmark_genes in benchmarks.items():
                    genes_pool, status = get_model_gene_pool(
                        model_name=model_name,
                        config=config,
                        tissue_label=tissue_label,
                        tissue_dir=tissue_dir,
                        threshold=threshold,
                        header_genes=header_genes,
                    )
                    if status == 1:
                        insufficient_set.add((tissue_label, threshold))
                        continue
                    if status == 2 or not genes_pool:
                        continue

                    if len(genes_pool) == threshold:
                        selected = set(genes_pool)
                        precision, tp = calculate_precision(selected, benchmark_genes)
                        n_selected = len(selected)
                    else:
                        seed = stable_seed("human_precision", model_name, tissue_label, threshold, benchmark_name)
                        rng = random.Random(seed)
                        precision_scores, tp_scores = [], []
                        for _ in range(NUM_RANDOM_SAMPLES):
                            sampled = set(rng.sample(genes_pool, threshold))
                            current_precision, current_tp = calculate_precision(sampled, benchmark_genes)
                            precision_scores.append(current_precision)
                            tp_scores.append(current_tp)
                        precision = float(np.mean(precision_scores))
                        tp = int(round(np.mean(tp_scores)))
                        n_selected = threshold

                    all_raw_results.append({
                        "Model": model_name,
                        "Benchmark_Name": benchmark_name,
                        "Tissue": tissue_label,
                        "Run": f"Fold_{TARGET_RUN_IDX}" if model_name != "sage" else SAGE_TARGET_FOLD,
                        "Threshold": threshold,
                        "Predicted_Genes_Count": n_selected,
                        "True_Positives_Count": tp,
                        "Precision": precision,
                    })
                    successfully_processed.add((model_name, tissue_label, threshold))
                    if not pool_cached:
                        prediction_pools_cache[(model_name, tissue_label, threshold)] = genes_pool
                        pool_cached = True

    print("\n--- Stage 2: 按组织并行进行 mlp_tanh 年龄预测 ---")
    tissue_tasks = [(label_to_dir[label], label) for label in tissue_labels if any(k[1] == label for k in prediction_pools_cache)]
    age_result_list = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
        delayed(process_one_tissue_age_prediction)(
            tissue_dir, tissue_label, prediction_pools_cache, insufficient_set, gene_to_idx
        )
        for tissue_dir, tissue_label in tissue_tasks
    )

    age_prediction_results = {}
    for item in age_result_list:
        age_prediction_results.update(item)

    for row in all_raw_results:
        key = (row["Model"], row["Tissue"], row["Threshold"])
        row.update(age_prediction_results.get(key, {"Pearson_R": np.nan, "MAE": np.nan, "RMSE": np.nan}))


    num_total_models = len(MODEL_CONFIGS)
    model_counts = {}
    for model, tissue, threshold in successfully_processed:
        model_counts.setdefault((tissue, threshold), set()).add(model)

    final_tissue_thresholds = set()
    for key, models in model_counts.items():
        if len(models) == num_total_models and key not in insufficient_set:
            final_tissue_thresholds.add(key)

    filtered_results = [r for r in all_raw_results if (r["Tissue"], r["Threshold"]) in final_tissue_thresholds]
    results_df = pd.DataFrame(filtered_results)
    if not results_df.empty:
        summary_df = (
            results_df.groupby(["Model", "Benchmark_Name", "Tissue", "Threshold"], observed=True)[
                ["Precision", "Pearson_R", "MAE", "RMSE"]
            ].mean().reset_index()
        )
    else:
        summary_df = pd.DataFrame()

    full_path = OUTPUT_DIR / "human_benchmark_mlp_tanh_prediction.csv"
    summary_path = OUTPUT_DIR / "human_benchmark_mlp_tanh_prediction_summary.csv"
    results_df.to_csv(full_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    pd.DataFrame(sorted(insufficient_set), columns=["Tissue", "Threshold"]).to_csv(
        OUTPUT_DIR / "human_benchmark_mlp_tanh_insufficient_gene_sets.csv", index=False
    )
    pd.DataFrame(
        [{"Model": m, "Tissue": t, "Threshold": th} for m, t, th in sorted(successfully_processed)]
    ).to_csv(OUTPUT_DIR / "human_benchmark_mlp_tanh_successfully_processed.csv", index=False)

    print("\n完成。")
    print(f"Full results: {full_path}")
    print(f"Summary: {summary_path}")
    print(f"Output dir: {OUTPUT_DIR}")
    return results_df, summary_df


if __name__ == "__main__":
    human_mlp_results_df, human_mlp_summary_df = main()
    display(human_mlp_results_df)
    display(human_mlp_summary_df)
